## Preprocesamiento y Modelado

Una vez inspeccionado el dataset en `customer_churn_eda.ipynb` definimos una una estrategia de preprocesamiento iterativo (de menos a más) para el encontrar

In [104]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import OneHotEncoder,RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, make_scorer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.discriminant_analysis import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from feature_engine.imputation import RandomSampleImputer
from xgboost import XGBClassifier


### Configuración de constantes, rutas y variables 

En esta sección definimos constantes, rutas de archivos y atributos del dataset

In [105]:
# Rutas de los archivos de datos
TRAIN_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/train.csv"
TEST_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Cargamos los datos
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Definir variables objetivo
TARGET = 'Exited'
# Variables numéricas: Incluyo las continuas y las binarias numéricas (HasCrCard, IsActiveMember)
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
# Variables categóricas: Las de texto con pocas categorías
CAT_FEATURES = ['Geography', 'Gender', 'Surname']
# Variables a eliminar inicialmente (IDs y apellido)
DROP_FEATURES = ['CustomerId']
SURNAME_COL = 'Surname'
RANDOM_STATE = 100            # Semilla para reproducibilidad

### 1. Preparación de Datos

Separamos variables independientes y dependientes en X_train e y_train por convención.

In [106]:
# X_train = variables independientes
# y_train = variable dependiente u objetivo
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

---------------------------------------

### Funciones auxiliar
#### Construir preprocesadores

In [ ]:
from sklearn.preprocessing import PowerTransformer


def make_preprocessor(X: pd.DataFrame, version: str):
    """Genera un ColumnTransformer con pipelines de preprocesamiento
    para variables numéricas y categóricas según la versión indicada.

    Args:
        X (pd.DataFrame): _input data frame_
        version (str): Versión del preprocesamiento en formato 'N#_C#'
        e.g. 'N3_C1' donde N# indica la versión numérica y C# la categórica
        numerical_cols (_type_): columnas numéricas para el preprocesamiento
        categorical_cols (_type_): columnas categóricas para el preprocesamiento

    Raises:
        ValueError: _unknown numeric version_
        ValueError: _unknown categorical version_

    Returns:
        _type_: ColumnTransformer con pipelines de preprocesamiento
    """
    num_version, categorical_version = version.split("_")  # e.g. 'N3', 'C1'

    # --- Numerical pipeline ---
    numerical_transformers = []

    if num_version == "N1":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con mediana
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N2":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con mediana + indicador de faltantes
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N3":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana + indicador de faltantes + escalado estandar
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N4":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con mediana + escalado estandar
        # Escalado robusto a outliers con RobustScaler
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N5":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        num_pipe = Pipeline(steps=[
            ('scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform"))
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N6":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        # Escalado robusto a outliers con RobustScaler
        num_pipe = Pipeline(steps=[
            ('pre_scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform")),
            ("scaler", RobustScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N7":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado RobustScaler
        # KNN requiere escalar primero para calcular distancias bien.
        num_pipe = Pipeline(steps=[
            ('scaler', RobustScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform")),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    
    elif num_version == "N8":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        num_pipe = Pipeline(steps=[
            ('scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=9, weights="uniform"))
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N9":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        # añadir indicador puede ayudar si hubiera correlación entre faltantes y target
        num_pipe = Pipeline(steps=[
            ('pre_scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform",add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N10":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        # vecinos más cercanos pesan más puede que la imputación sea más fina (si hay grupos claros)
        num_pipe = Pipeline(steps=[
            ('pre_scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="distance",add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N11":
        # Imputamos cada variable numérica por separado según su distribución
        # CreditScore= 20.51% missing -> Distribución normal -> SimpleImputer mediana
        # Age= 0.00% missing -> Distribución normal -> SimpleImputer mediana
        # Tenure= 0.00% missing -> Distribución plana -> RandomSampleImputer (de las observaciones existentes)
        # Balance= 20.19% missing -> Distribución binomial -> SimpleImputer constante 0 (asumimos que falta significa balance 0,abre cuenta sin saldo)
        # NumOfProducts= 12.20% missing -> SimpleImputer constante 0 (asumimos que falta significa no tener productos,abre cuenta sin productos)
        # EstimatedSalary= 10.40% missing -> Distribución plana -> RandomSampleImputer (de las observaciones existentes) 
        # Surname= 5.11% missing -> Categorical -> SimpleImputer unknown + OneHotEncoder
        # HasCrCard= 4.69% missing -> SimpleImputer constante 0 (asumimos que falta significa no tener tarjeta,abre cuenta sin solicitar tarjeta)
        # IsActiveMember= 0% missing -> Binaria -> SimpleImputer constante 0 (asumimos que falta significa no ser miembro activo)
        num_pipe_median = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")), # imputación mediana
            ("scaler", RobustScaler()), # escalado robusto a outliers
        ])
        
        num_pipe_random_sample = Pipeline(steps=[
            # random_state : int, variable name or a list of variables to determine the seed, observation per observation.
            #seed : 'general' (one seed will be used to impute the entire dataframe) or 'observation' (the seed will be set for each observation using the values of the variables indicated)
            ("imputer", RandomSampleImputer(random_state=RANDOM_STATE,seed='general')), # imputación por muestra aleatoria
            ("scaler", RobustScaler()), # escalado robusto a outliers
        ])
        num_pipe_constant_0 = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ])
        numerical_transformers.append(("median", num_pipe_median, ["CreditScore","Age"]))
        numerical_transformers.append(("random_sample", num_pipe_random_sample, ["Tenure","EstimatedSalary"]))
        numerical_transformers.append(("constant_0", num_pipe_constant_0, ["Balance","NumOfProducts","HasCrCard","IsActiveMember"]))
    elif num_version == "N12":
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con mediana + indicador de faltantes + PowerTransformer + escalado robusto
        # PowerTransformer con método Yeo-Johnson para variables con ceros o negativos
        # Kaggle scoring =0.57
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("power", PowerTransformer(method="yeo-johnson")),
            ("scaler", RobustScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N13":
        # Separamos numéricas en grupos: continuas, discretas, binarias
        # Variables continuas: Continuas: median + indicator + power + robust
        numerical_continuous_cols = ['CreditScore', 'Age', 
                                     'Balance', 'EstimatedSalary']
        # Variables discretas (enteras): Imputar most_frequent + standard scaler (opcional)
        numerical_discrete_cols = ['Tenure', 'NumOfProducts']
        # Variables binarias: imputar most_frequent (0/1) + estandar scaler (opcional)
        numerical_binary_cols = ['HasCrCard', 'IsActiveMember']
        # Pipeline para variables continuas
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante que puede ayudar si hubiera correlación entre faltantes y target
        cont_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("power", PowerTransformer(method="yeo-johnson")),
            ("scaler", RobustScaler()),
            #("scaler", StandardScaler()),  # Probar StandardScaler en continuas
        ])
        
        # Pipeline para variables discretas
        disc_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent",add_indicator=True)),
            #("scaler", StandardScaler()), # Escalar discretas no tiene sentido
        ])
        
        # Pipeline para variables binarias
        binary_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent",add_indicator=True)),
            #("scaler", StandardScaler()), # escalar binarias es absurdo
        ])
        numerical_transformers.append(("cont", cont_pipe, numerical_continuous_cols))
        numerical_transformers.append(("disc", disc_pipe, numerical_discrete_cols))
        numerical_transformers.append(("bin", binary_pipe, numerical_binary_cols))
        # NOTA: N13 aplica escalado distinto por grupos
    elif num_version == "N14":
        # Variables continuas: imputer+indicator + power
        # Variables discretas/binarias: imputer+indicator (sin power)
        # juntar y aplicar un único escalado al bloque numérico completo
        numerical_continuous_cols = ['CreditScore', 'Age', 
                                     'Balance', 'EstimatedSalary']
        numerical_discrete_binary_cols = ['Tenure', 'NumOfProducts']
        numerical_binary_cols = ['HasCrCard', 'IsActiveMember']
        # Pipeline para variables continuas
        cont_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("power", PowerTransformer(method="yeo-johnson")),
        ])
        # Pipeline para variables discretas + binarias
        disc_bin_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median",add_indicator=True)),
        ])
        # Transformer para juntar ambos grupos y escalar
        num_split = ColumnTransformer(
            transformers=[
                ("cont", cont_pipe, numerical_continuous_cols),
                ("disc_bin", disc_bin_pipe, numerical_discrete_binary_cols + numerical_binary_cols),
            ],
            remainder="drop",
            sparse_threshold=0.0
        )
        # Pipeline final numérico con escalado tras juntar
        num_pipe = Pipeline([
            ("num_split", num_split),
            ("scaler", RobustScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, 
                                       numerical_continuous_cols + 
                                       numerical_discrete_binary_cols + 
                                       numerical_binary_cols))

    else:
        raise ValueError(f"Unknown numeric version: {num_version}")

    # --- Categorical pipeline (si aplica) ---
    categorical_transformers = []
    if categorical_version == "C0": 
        # No aplica pipeline en categoricas
        pass
    elif categorical_version == "C1":
        # Categóricas normales: imputar + onehot
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        categorical_transformers.append(("cat", categorical_pipe, ['Geography', 'Gender']))
    elif categorical_version == "C11":
        # Continuación de num_version == "N11"
        # Surname= 5.11% missing -> Categorical -> SimpleImputer unknown + OneHotEncoder
        #print("Categorical version C11: special handling for Surname column")
        cat_pipe_surname = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ])
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        categorical_transformers.append(("surname", cat_pipe_surname, ["Surname"]))
        categorical_transformers.append(("cat", categorical_pipe, ["Geography", "Gender"]))
    elif categorical_version == "C111":
        # Categoricas normales: Imputar most_frequent + onehot sparse_output=False
        # Sparse_output=False para devolver array denso (no sparse)
        categorical_cols = ['Geography', 'Gender']
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])
        categorical_transformers.append(("cat", categorical_pipe, categorical_cols))

    else:
        raise ValueError(f"Unknown categorical version: {categorical_version}")

    # --- Build ColumnTransformer ---
    # Transformers numéricos y categóricos
    transformers = []
    transformers.extend(numerical_transformers)     
    transformers.extend(categorical_transformers)

    # Construimos el ColumnTransformer final
    # que une pipelines numéricos + pipelines categóricos
    col_trans_preprocessor = ColumnTransformer(
        transformers=transformers, # lista de tuplas (name, pipeline, cols)
        remainder="drop", # elimina columnas no especificadas
        verbose_feature_names_out=True) # nombres detallados de columnas
    return col_trans_preprocessor


#### Construir pipelines

In [108]:
# Creación y evaluación del pipeline
# Construcción del pipeline con preprocesador y modelo

def make_pipeline(preprocessor: ColumnTransformer, model=None):
    """Construye un Pipeline con el preprocesador y el modelo indicado.
    Args:
        preprocessor (ColumnTransformer): Preprocesador ColumnTransformer
        model (_type_, optional): Modelo de clasificación. Defaults to None.
    Returns:
        Pipeline: Pipeline con preprocesador y modelo, si no se indica modelo
        se usa LinearDiscriminantAnalysis por defecto.
    """
    if model is None:
        model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        #model = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE,class_weight='balanced',n_jobs=-1)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])




#### Evaluar pipelines

Esta función evalua el pipeline usando validación cruzada estratificada

In [109]:
def evaluate_pipeline(pipe: Pipeline, X: pd.DataFrame, y: pd.Series, n_splits=5):
    """ Evalúa el pipeline usando Validación Cruzada estratificada
    Args:
        pipe (Pipeline): Pipeline a evaluar.
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        dict: Diccionario con las métricas promedio y desviación estándar.
    """
    # Configuramos la Validación Local Cruzada  n_splits splits (divisiones)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    # Definimos las métricas que queremos extraer
    # f1, roc_auc, precision, recall, accuracy son strings estándar de sklearn.
    # Kappa requiere make_scorer.
    scoring_metrics = {
        "f1": "f1",
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        'kappa': make_scorer(cohen_kappa_score),
        'precision': 'precision',
        'recall': 'recall',
    }
    cv_results = cross_validate(pipe, X, y, cv=cv, scoring=scoring_metrics, n_jobs=-1)
    return {
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std":  cv_results["test_f1"].std(),
        "auc_mean": cv_results["test_roc_auc"].mean(),
        "auc_std":  cv_results["test_roc_auc"].std(),
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std":  cv_results["test_accuracy"].std(),
        "kappa_mean": cv_results["test_kappa"].mean(),
        "kappa_std":  cv_results["test_kappa"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std":  cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std":  cv_results["test_recall"].std()
    }

#### Evaluar varios modelos

In [110]:

def benchmark_models_with_fixed_preprocess(X: pd.DataFrame, y: pd.Series, models: dict, 
                                           best_preprocesor_version: str, n_splits=5):
    """Evalúa varios modelos con un preprocesador fijo usando Validación Cruzada.
    Args:
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        models (dict): Diccionario con nombre y objeto del modelo a evaluar.
        best_preprocesor_version (str): Versión del preprocesador a usar.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        pd.DataFrame: DataFrame con resultados de cada modelo evaluado.
    """
    # Construimos el preprocesador fijo con la mejor versión
    best_preprocesor = make_preprocessor(X, best_preprocesor_version)

    model_results = []  # Lista para almacenar resultados de cada modelo
    # Evaluamos cada modelo con el preprocesador fijo
    for name, model in models.items():
        # Construimos el pipeline con preprocesador fijo y el modelo actual
        pipe = Pipeline([("preprocessor", best_preprocesor), ("classifier", model)])
        try:
            # Evaluamos el pipeline con validación cruzada para el pipeline actual
            cv_metrics = evaluate_pipeline(pipe, X, y , n_splits=n_splits)
            model_results.append({
                "model": name,
                "preprocessor_version": best_preprocesor_version,
                **cv_metrics
            })
        except Exception as e:
            model_results.append({"model": name, "error": str(e)})
    # Construimos el DataFrame de resultados ordenado por F1 medio
    out = pd.DataFrame(model_results).sort_values(by="f1_mean", ascending=False, na_position="last")
    return out


------------------
## Evaluación de diferentes preprocesadores y modelos

A partir de aquí comenzamos la evaluación de los distintos preprocesadores que se han configurado en la función `make_preprocesor` y modelos, configurador `benchmark_models_with_fixed_preprocess`

In [ ]:
# Experimentos: combinaciones de preprocesamiento a probar

EXPERIMENTS = [
    #"N1_C0",  # num only (simple imputer mediana)
    #"N2_C0",  # num only (simple imputer mediana) + indicator
    #"N3_C0",  # num only (simple imputer mediana) + indicator + scaler
    #"N3_C1",  # num only (simple imputer mediana + indicator + scaler) + cat (onehot) sin surname
    #"N4_C1",  # num (robust scaler) + cat (onehot) sin surname
    #"N5_C1",  # num (KNN imputer 5 vecinos) + cat (onehot) sin surname
    #"N6_C1",  # num (KNN imputer + robust scaler) + cat (onehot) sin surname
    #"N7_C1",  # num (KNN imputer + robust scaler) + cat (onehot) sin surname
    #"N8_C1",  # num (KNN imputer 9 vecinos) + cat (onehot) sin surname
    # No mejora -> Volvemos a la base N5_C1
    #"N9_C1",  # num (KNN imputer 5 vecinos + indicador) + cat (onehot) sin surname
    #"N10_C1",  # num (KNN imputer 5 vecinos + indicador + pesos distancia) + cat (onehot) sin surname
    #"N11_C1",  # num (imputación por variable según distribución + RandomSampleImputer) + cat (surname imputación unknown + onehot)
    #"N12_C1",  # num (imputación mediana + indicador + PowerTransformer + RobustScaler) + cat (onehot) sin surname
    #"N13_C1",  # num en grupos: continuas, discretas, binarias + cat (onehot) sin surname
    #"N13_C111",# num en grupos: continuas, discretas, binarias + cat (onehot sparse_output=False)
    "N14_C1",  # num en grupos: continuas (power), discretas, binarias escalado final+ cat (onehot) sin surname
]

# Modelos a probar
models = {
    # Modelos lineales
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"),
    # Arboles
    # No necesitan escalado de variables
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced",n_jobs=-1,),
    "RandomForest_bal_subsample": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced_subsample",n_jobs=-1,),
    # Otros modelos
    "NaiveBayes": GaussianNB(),
    "RedesNeurales": MLPClassifier(hidden_layer_sizes=(50,30), max_iter=1000, random_state=RANDOM_STATE,
                                   activation='relu',solver='adam',early_stopping=True),
    "KNN_5": KNeighborsClassifier(n_neighbors=5, n_jobs=-1,),
    "KNN_5_distance": KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),

    
}


#### Búsqueda del mejor pipeline

Evaluamos las diferentes configuraciones de preprocesador que tenemos configuradas con el modelo por defecto definido en la función `make_pipeline` con el objetivo de obtener mejor versión o combinación de preprocesado. 

In [112]:
preprocesors_results = []
for experiment_version in EXPERIMENTS:
    best_preprocesor = make_preprocessor(train_df, experiment_version)
    
    pipe = make_pipeline(best_preprocesor) 
    print("Evaluando preprocesador versión:", experiment_version)
    cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
    preprocesors_results.append({"version": experiment_version, **cv_metrics})
    #print(experiment_version, cv_metrics)

results_df = pd.DataFrame(preprocesors_results).sort_values("f1_mean", ascending=False)
display(results_df)
best_preprocesor_version = results_df.iloc[0]["version"]
print("Mejor versión de preprocesador:", best_preprocesor_version)

Evaluando preprocesador versión: N3_C1
Evaluando preprocesador versión: N4_C1
Evaluando preprocesador versión: N5_C1
Evaluando preprocesador versión: N6_C1
Evaluando preprocesador versión: N12_C1
Evaluando preprocesador versión: N13_C1
Evaluando preprocesador versión: N13_C111
Evaluando preprocesador versión: N14_C1


,version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
4,N12_C1,0.354547,0.017119,0.777805,0.008821,0.814875,0.004700,0.268236,0.018070,0.613039,0.030478,0.249693,0.014851
2,N5_C1,0.332000,0.017123,0.770809,0.008083,0.808500,0.005056,0.242826,0.017811,0.575508,0.034044,0.233742,0.015325
3,N6_C1,0.332000,0.017123,0.770809,0.008083,0.808500,0.005056,0.242826,0.017811,0.575508,0.034044,0.233742,0.015325
7,N14_C1,0.331665,0.018189,0.771331,0.008939,0.813375,0.002669,0.248997,0.016260,0.613674,0.019473,0.227607,0.016846
5,N13_C1,0.330955,0.017166,0.771344,0.008929,0.813250,0.002634,0.248292,0.015361,0.613095,0.019739,0.226994,0.015880
6,N13_C111,0.330955,0.017166,0.771344,0.008929,0.813250,0.002634,0.248292,0.015361,0.613095,0.019739,0.226994,0.015880
0,N3_C1,0.328250,0.017747,0.768164,0.008146,0.807250,0.005585,0.238353,0.018927,0.567959,0.036011,0.231288,0.015350
1,N4_C1,0.324909,0.016104,0.768646,0.008239,0.806875,0.005506,0.235328,0.017407,0.566711,0.036724,0.228221,0.014203


Mejor versión de preprocesador: N12_C1


#### Búsqueda del mejor modelo

Evaluamos los modelos con el mejor preprocesador encontrado 

In [113]:
# Evaluamos los modelos con el mejor preprocesador encontrado
models_df = benchmark_models_with_fixed_preprocess(X_train, y_train, models, best_preprocesor_version, n_splits=5)
print("---- Resultados de validación cruzada de modelos con preprocesador fijo:----")
display(models_df)

best_model_name = models_df.iloc[0]["model"]
print("Mejor versión de preprocesador:", best_preprocesor_version)
print("Mejor modelo:", best_model_name)

---- Resultados de validación cruzada de modelos con preprocesador fijo:----


,model,preprocessor_version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
4,RandomForest_bal_subsample,N12_C1,0.527004,0.017626,0.833549,0.009624,0.850000,0.004348,0.446373,0.018888,0.736969,0.018768,0.410429,0.018446
3,RandomForest,N12_C1,0.523368,0.021228,0.833931,0.009506,0.849000,0.004978,0.442303,0.022521,0.733027,0.020271,0.407362,0.022575
6,RedesNeurales,N12_C1,0.510927,0.042359,0.823820,0.018407,0.841000,0.011677,0.423372,0.046848,0.684975,0.050216,0.409202,0.046424
1,LogisticRegression,N12_C1,0.499774,0.011828,0.778788,0.009425,0.712000,0.008628,0.321042,0.016159,0.386878,0.010001,0.706135,0.022154
2,DecisionTree,N12_C1,0.469321,0.023509,0.666476,0.014370,0.785000,0.012393,0.334605,0.031049,0.473057,0.029824,0.466258,0.022708
5,NaiveBayes,N12_C1,0.433126,0.019169,0.753873,0.012791,0.796250,0.010406,0.311701,0.026265,0.501574,0.034606,0.381595,0.013101
8,KNN_5_distance,N12_C1,0.381314,0.019992,0.721965,0.006863,0.806625,0.005750,0.279081,0.022109,0.547932,0.027540,0.292638,0.017949
7,KNN_5,N12_C1,0.378233,0.013446,0.719038,0.006674,0.809000,0.002894,0.279372,0.013683,0.561572,0.013811,0.285276,0.013014
0,LinearDiscriminantAnalysis,N12_C1,0.356414,0.017484,0.777760,0.008804,0.815500,0.004444,0.270469,0.018052,0.616789,0.028319,0.250920,0.015447


Mejor versión de preprocesador: N12_C1
Mejor modelo: RandomForest_bal_subsample


## Construcción del pipeline final para Kaggle

Una vez obtenido la mejor combinación de preprocesadores y el mejor modelo, construimos el pipeline final para kaggle con la mejor combinación de ambos y volvemos a ejecutar la evaluación y el entrenamiento para finalmente obtener la predicción y generar el fichero para kaggle. 

In [114]:
# Construcción del pipeline final para Kaggle con el mejor preprocesador y modelo
# Construimos el preprocesador fijo con la mejor versión
best_preprocesor = make_preprocessor(X_train, best_preprocesor_version)
best_model = models[best_model_name]
# Pipeline Completo (Preprocesamiento + Modelo)
best_model_pipeline = Pipeline(steps=[
    ('preprocessor', best_preprocesor),
    ('classifier', best_model)
])
# Configuramos y ejecutamos la Validación Cruzada local
cv_metrics = evaluate_pipeline(best_model_pipeline, X_train, y_train, n_splits=5)
# Generación de Submission para Kaggle con el mejor modelo encontrado
# Re-entrenamos con TODOS los datos de train para la predicción final
best_model_pipeline.fit(X_train, y_train) 
test_predictions = best_model_pipeline.predict(test_df)

# Crear fichero de salida
submission_df = pd.DataFrame({
    'CustomerId': test_df['CustomerId'],
    'Exited': test_predictions
})
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Fichero '{SUBMISSION_PATH}' generado correctamente.")

print("\n---- Mejores Resultados y Validación Cruzada local -----")
print("Mejor modelo:", best_model_name)
print("Mejor versión de preprocesador:", best_preprocesor_version)
print(f"Mean F1-Score:  {cv_metrics['f1_mean']:.4f} (+/- Std {cv_metrics['f1_std']:.4f})")
print(f"Mean Accuracy:  {cv_metrics['accuracy_mean']:.4f} (+/- Std {cv_metrics['accuracy_std']:.4f})")
print(f"Mean Kappa:     {cv_metrics['kappa_mean']:.4f}")
print(f"Mean Precision: {cv_metrics['precision_mean']:.4f}")
print(f"Mean Recall:    {cv_metrics['recall_mean']:.4f}")



Fichero '/kaggle/working/submission.csv' generado correctamente.

---- Mejores Resultados y Validación Cruzada local -----
Mejor modelo: RandomForest_bal_subsample
Mejor versión de preprocesador: N12_C1
Mean F1-Score:  0.5270 (+/- Std 0.0176)
Mean Accuracy:  0.8500 (+/- Std 0.0043)
Mean Kappa:     0.4464
Mean Precision: 0.7370
Mean Recall:    0.4104
